In [ ]:
import os, sys
sys.path.append("../")
import numpy as np
import matplotlib.pyplot as plt

from lowRank.NMLRSSE_Strang_CUDA import NMLRSSE_Strang_CUDA
from lowRank import compute_expectation as ce
from utils.noise_generator import ColoredNoiseGenerator_Cholesky
from utils import multi_index

# SIAM-style plotting setup: clean, publication-friendly, and readable in grayscale.
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.family": "serif",
    "mathtext.fontset": "stix",
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 10,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "xtick.minor.size": 2.5,
    "ytick.minor.size": 2.5,
    "legend.fontsize": 9,
    "legend.frameon": False,
})

# bath correlation function in Cai_2020_CPAM
Delta = 1
beta = 5/Delta
wc = 2.5 * Delta
wmax = 4 * wc
CapL = 200
factor = 1 - np.exp(-wmax/wc)
wl = -wc * np.log(1 - np.linspace(1, CapL, CapL)/CapL * factor)
cl = wl * np.sqrt((0.2 * wc/CapL) * factor)

_coth = 1.0 / np.tanh(0.5 * beta * wl)
_pref = (cl**2) / (2.0 * wl)
def bath_corr(t):
    t_arr = np.asarray(t)
    t1 = np.atleast_1d(t_arr).astype(float)
    phase = wl[:, None] * t1[None, :]             # shape: (CapL, len(t))
    out = np.sum(_pref[:, None] * (np.cos(phase) * _coth[:, None] - 1j * np.sin(phase)), axis=0)
    return out[0] if t_arr.ndim == 0 else out

# spin-boson model parameters
eps = [0, Delta, 2*Delta]
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
Hs = [Delta*sx + e*sz for e in eps]
L = sz

## Test: BCF with Ohmic spectrum

In [ ]:
# parameters for the simulation
tmax = 5.0 / Delta
N_steps = 100
psi0 = np.array([1.0, 0.0], dtype=complex)
N_trajs = [100, 1000, 10000]
rank = 10
# hierarchy truncation (total order cutoff for the multi-index set)
# set smaller than N_steps to avoid combinatorial blow-up when rank grows
max_layer = 8
parallel_traj = True

# Save trajectories for each eps when N_traj reaches this target.
SAVE_TRAJ = 10000
SAVE_DIR = "test_spinboson"
os.makedirs(SAVE_DIR, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(11.2, 3.4), sharey=True)
panel_labels = ["(a)", "(b)", "(c)"]
color = ["tab:blue", "tab:orange", "tab:green"]

for idx in range(3):
    NMLRSSE_solver = NMLRSSE_Strang_CUDA(
        Hs = Hs[idx],
        L = L,
        bath_corr = bath_corr,
        tmax = tmax,
        N_steps = N_steps,
        rank = rank,
        max_layer = max_layer,
    )

    for id, N_traj in enumerate(N_trajs):
        psis = NMLRSSE_solver.solve(N_traj, psi0)
        sigma_z = ce.compute_expectation_value_linear(psis, sz)

        # Persist the full 10000-trajectory result for each eps case.
        if N_traj == SAVE_TRAJ:
            eps_val = eps[idx]
            fname = (
                "strangcuda_spinboson_"
                f"tmax{tmax:g}_Nsteps{N_steps}_rank{rank}_maxlayer{max_layer}_"
                f"Ntraj{N_traj}_eps{eps_val:g}.npz"
            )
            fpath = os.path.join(SAVE_DIR, fname)
            np.savez(
                fpath,
                psis=psis,
                sigma_z=sigma_z,
                t_grid=NMLRSSE_solver.t_grid,
                eps=eps_val,
                tmax=tmax,
                N_steps=N_steps,
                rank=rank,
                max_layer=max_layer,
                N_traj=N_traj,
            )
            print(f"Saved 10000-traj result: {fpath}")

        # Main method curve and reference with distinct styles for print clarity.
        axes[idx].plot(
            NMLRSSE_solver.t_grid, sigma_z,
            color=color[id], lw=1.9,
            label="{} trajs".format(N_traj)
        )

    data = np.load("ref_spinboson/eps_{}.npz".format(eps[idx]))
    tgrid = data['t']
    sigmaz = data['sz']
    axes[idx].plot(
        tgrid, sigmaz,
        color="black", lw=1.3, ls="--",
        label="TEMPO"
    )

    axes[idx].set_xlabel(r"$t$")
    axes[idx].set_ylabel(r"$\langle \sigma_z \rangle$")
    axes[idx].set_title(r"{}  $\epsilon={}$".format(panel_labels[idx], eps[idx]))

    axes[idx].minorticks_on()
    axes[idx].grid(which="major", alpha=0.28, lw=0.6)
    axes[idx].grid(which="minor", alpha=0.12, lw=0.4)
    axes[idx].legend(loc="best")

    test_op = sigma_z[::(N_steps//50)]
    error = np.linalg.norm(test_op - sigmaz) / np.linalg.norm(sigmaz)
    print("Relative error for eps={}: {:.3e}".format(eps[idx], error))

fig.suptitle(
    r"Spin-boson dynamics: rank$=%d$, max layer$=%d$" % (rank, max_layer),
    y=1.03, fontsize=11
)
plt.tight_layout()
plt.show()

## Test: A simple BCF

In [ ]:
tmax = 2
N_steps = 100
t_grid = np.linspace(0, tmax, N_steps + 1)
psi0 = np.array([1.0, 1j]) / np.sqrt(2)
N_trajs = [100, 1000, 10000]
rank = 1
max_layer = 8
parallel_traj = True

eps = 1
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
sig = [sx, sy, sz]
sig_ylabel = [r"$\langle \sigma_x \rangle$", r"$\langle \sigma_y \rangle$", r"$\langle \sigma_z \rangle$"]
Hs = eps / 2 * sz
lamda = np.sqrt(2)
gamma = 1
L = lamda * sz

F = lambda t : 1j*eps*t + 2*lamda**2/gamma * (np.exp(-gamma*t)+gamma*t-1)
rho_init = np.outer(psi0, np.conj(psi0))
rho = lambda t : np.array([[rho_init[0,0], rho_init[0,1]*np.exp(-F(t))],
                           [rho_init[1,0]*np.exp(-np.conj(F(t))), rho_init[1,1]]], dtype=complex)
ref_rho = np.array([rho(t) for t in t_grid])

def bath_corr(t):
    t_arr = np.asarray(t)
    t1 = np.atleast_1d(t_arr).astype(float)
    out = gamma/2 * np.exp(-gamma * np.abs(t1))
    return out[0] if t_arr.ndim == 0 else out

lam = gamma / 2
U = np.exp(-lam * t_grid)
U = U[:, np.newaxis]
V = np.exp(lam * t_grid)
V = V[np.newaxis, :]

fig, axes = plt.subplots(1, 3, figsize=(11.2, 3.4), sharey=True)
panel_labels = ["(a)", "(b)", "(c)"]
color = ["tab:blue", "tab:orange", "tab:green"]

NMLRSSE_solver = NMLRSSE_Strang_CUDA(
    Hs = Hs,
    L = L,
    bath_corr = bath_corr,
    tmax = tmax,
    N_steps = N_steps,
    rank = rank,
    max_layer = max_layer,
    do_low_rank_decomposition=False,
)
NMLRSSE_solver.set_lamUV(lam, U, V)

for ik, N_traj in enumerate(N_trajs):
    psis = NMLRSSE_solver.solve(N_traj, psi0)
    
    for id in range(3):
        sigma_expect = ce.compute_expectation_value_linear(psis, sig[id])
        axes[id].plot(
            NMLRSSE_solver.t_grid, sigma_expect,
            color=color[ik], lw=1.9,
            label="{} trajs".format(N_traj)
        )

for idx in range(3):
    ref_sig = np.array([np.trace(ref_rho[i] @ sig[idx]) for i in range(len(t_grid))])
    axes[idx].plot(
        t_grid, ref_sig,
        color="black", lw=1.3, ls="--",
        label="TEMPO"
    )
    axes[idx].set_xlabel(r"$t$")
    axes[idx].set_ylabel(sig_ylabel[idx])
    axes[idx].minorticks_on()
    axes[idx].grid(which="major", alpha=0.28, lw=0.6)
    axes[idx].grid(which="minor", alpha=0.12, lw=0.4)
    axes[idx].legend(loc="best")